# ForestGuard missing 2022 observation
Private Kaggle CPU: Accelerator None, Internet On. One 1 October 2022 scene reprocessed in 2024.
Read bounded 20m crops; calibrate from original product XML (quantification and band offsets), require unscaled COG headers; mask uncertain pixels. Public read token stays in runtime memory.
Existing trusted weak-map model only; no training. Export source XML, masks, source IDs, checksums and ZIP. The estimate is not independently validated forest cover.


In [ ]:
import base64


In [ ]:
STUDY=None
MODEL_SHA='8bf0858faaee969b35e003c466de19ab1767a96c5c8ed9098cb8010f13d3a6b8'
MODEL_BYTES=base64.b64decode(None) if False else b""


In [ ]:
"""Numerical research features; these functions do not generate land-cover labels."""
import numpy as np


def research_features(values, band_order, usable):
    values = np.asarray(values, dtype='float32')
    usable = np.asarray(usable, dtype=bool)
    if values.ndim != 3 or values.shape[1:] != usable.shape or values.shape[0] != len(band_order):
        raise ValueError('Band stack and usable mask must share a grid.')
    if len(set(band_order)) != len(band_order):
        raise ValueError('Band names must be unique.')
    required = {'B02','B04','B05','B08','B8A','B11'}
    if not required <= set(band_order):
        raise ValueError('Required visible/NIR/red-edge/SWIR bands are missing.')
    band = dict(zip(band_order, values))
    valid = usable & np.isfinite(values).all(axis=0)

    def ratio(numerator, denominator):
        result = np.full(usable.shape, np.nan, dtype='float32')
        np.divide(numerator, denominator, out=result, where=valid & (denominator > 1e-6))
        return result

    ndvi = ratio(band['B08']-band['B04'], band['B08']+band['B04'])
    ndre = ratio(band['B8A']-band['B05'], band['B8A']+band['B05'])
    ndmi = ratio(band['B8A']-band['B11'], band['B8A']+band['B11'])
    evi = ratio(2.5*(band['B08']-band['B04']), band['B08']+6*band['B04']-7.5*band['B02']+1)
    index_valid = valid & np.isfinite(ndvi) & (np.abs(ndvi) <= 1)
    for index in [ndre, ndmi]:
        index_valid &= np.isfinite(index) & (np.abs(index) <= 1)
    padded = np.pad(np.where(index_valid, ndvi, 0), 1)
    padded_valid = np.pad(index_valid, 1, constant_values=False)
    count = np.zeros(usable.shape, dtype='uint8')
    total = np.zeros(usable.shape, dtype='float32')
    squares = total.copy()
    height, width = usable.shape
    for dy in range(3):
        for dx in range(3):
            local = padded[dy:dy+height, dx:dx+width]
            count += padded_valid[dy:dy+height, dx:dx+width]
            total += local
            squares += local*local
    texture = np.sqrt(np.maximum(0, squares/9-(total/9)**2))
    features = np.concatenate([values, np.stack([ndvi,evi,ndre,ndmi,texture])])
    feature_valid = index_valid & (count == 9) & np.isfinite(features).all(axis=0)
    features[:, ~feature_valid] = np.nan
    return features, feature_valid, list(band_order)+['NDVI','EVI','NDRE_B8A_B05','NDMI_B8A_B11','NDVI_STD_3X3']


In [ ]:
"""Bounded hosted 2022 crop using original product XML calibration."""
import hashlib
import json
import math
import platform
import shutil
import xml.etree.ElementTree as ET
from datetime import datetime,timezone
from pathlib import Path
from urllib.request import urlopen

BAND_IDS={'B02':'1','B03':'2','B04':'3','B08':'7','B05':'4','B06':'5','B07':'6','B8A':'8','B11':'11','B12':'12'}
PC_ITEM=None


def calibration_xml(raw):
    if len(raw)>1024**2 or b'<!DOCTYPE' in raw.upper() or b'<!ENTITY' in raw.upper():raise ValueError('Unsafe product XML')
    root=ET.fromstring(raw)
    values={}
    offsets={}
    for element in root.iter():
        tag=element.tag.split('}')[-1]
        if tag in ['BOA_QUANTIFICATION_VALUE','PROCESSING_BASELINE']:values[tag]=element.text
        if tag=='BOA_ADD_OFFSET':offsets[element.attrib['band_id']]=float(element.text)
    quant=float(values['BOA_QUANTIFICATION_VALUE'])
    if not math.isfinite(quant) or quant<=0 or float(values['PROCESSING_BASELINE'])<5:raise ValueError('Require reprocessed Collection-1 metadata')
    if any(key not in offsets or not math.isfinite(offsets[key]) for key in BAND_IDS.values()):raise ValueError('Missing band offsets')
    return {band:{'scale':1/quant,'offset':offsets[key]/quant} for band,key in BAND_IDS.items()}


def run_2022(boundary,base_run,feature_builder):
    base=Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('/content')
    if platform.system()=='Windows' or not base.is_dir():raise RuntimeError('Use private Kaggle/Colab CPU runtime')
    if shutil.disk_usage(base).free<250*1024**2:raise ValueError('Insufficient cloud storage')
    if boundary is None or PC_ITEM is None:raise ValueError('Use prepared private notebook')
    import numpy as np
    import rasterio
    from rasterio.features import geometry_mask
    from rasterio.windows import from_bounds,Window
    from rasterio.warp import transform_bounds,transform_geom,Resampling
    from rasterio.vrt import WarpedVRT
    from PIL import Image
    item=PC_ITEM
    if item['id']!='S2A_MSIL2A_20221001T052651_R105_T43QFE_20240724T063709':raise ValueError('Unreviewed source scene')
    with urlopen('https://planetarycomputer.microsoft.com/api/sas/v1/token/sentinel2l2a01/sentinel2-l2',timeout=30) as response:token=json.loads(response.read(16385))['token']
    def signed(href):
        if not href.startswith('https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/') or '?' in href:raise ValueError('Unexpected asset URL')
        return href+'?'+token
    with urlopen(signed(item['assets']['product-metadata']['href']),timeout=30) as response:xml=response.read(1024**2+1)
    calibration=calibration_xml(xml)
    output=base/'forestguard_2022'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ');folder=output/'2022';folder.mkdir(parents=True)
    (output/'boundary.geojson').write_bytes((json.dumps(boundary,indent=2)+'\n').encode())
    (folder/'product.xml').write_bytes(xml)
    (folder/'source.json').write_bytes((json.dumps(item,indent=2)+'\n').encode())
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR',GDAL_HTTP_TIMEOUT=60,GDAL_CACHEMAX=64*1024**2):
        with rasterio.open(signed(item['assets']['B05']['href'])) as src:
            bounds=transform_bounds('EPSG:4326',src.crs,*boundary['bbox'])
            crop=from_bounds(*bounds,transform=src.transform)
            left,top=math.floor(crop.col_off),math.floor(crop.row_off)
            right,bottom=math.ceil(crop.col_off+crop.width),math.ceil(crop.row_off+crop.height)
            if not (src.res==(20.,20.) and 0<=left<right<=src.width and 0<=top<bottom<=src.height and max(right-left,bottom-top)<=256):raise ValueError('Invalid crop budget/grid')
            window=Window(left,top,right-left,bottom-top);grid=src.window_transform(window);crs=src.crs;shape=(bottom-top,right-left)
        arrays=[];valid=np.ones(shape,dtype=bool)
        for band in BAND_IDS:
            with rasterio.open(signed(item['assets'][band]['href'])) as src:
                expected=10. if band in ['B02','B03','B04','B08'] else 20.
                if src.crs!=crs or src.res!=(expected,expected):raise ValueError('Unexpected native band grid')
                # These COGs preserve SAFE digital numbers; XML supplies calibration.
                if (src.scales[0],src.offsets[0])!=(1.,0.):raise ValueError('Encoded calibration requires review before applying XML')
                with WarpedVRT(src,crs=crs,transform=grid,height=shape[0],width=shape[1],resampling=Resampling.average) as aligned:raw=aligned.read(1,masked=True).astype('float32')
            valid&=~np.ma.getmaskarray(raw)
            arrays.append(raw.filled(np.nan)*calibration[band]['scale']+calibration[band]['offset'])
        with rasterio.open(signed(item['assets']['SCL']['href'])) as src:
            if src.crs!=crs or src.res!=(20.,20.):raise ValueError('Invalid SCL grid')
            with WarpedVRT(src,crs=crs,transform=grid,height=shape[0],width=shape[1],resampling=Resampling.nearest) as aligned:scl=aligned.read(1)
    inside=geometry_mask([transform_geom('EPSG:4326',crs,boundary['features'][0]['geometry'])],shape,grid,invert=True)
    stack=np.stack(arrays);valid&=inside&np.isin(scl,[4,5,6])&np.isfinite(stack).all(axis=0)
    features,usable,names=feature_builder(stack,list(BAND_IDS),valid)
    fraction=float(usable.sum()/inside.sum())
    if fraction<.5:raise ValueError(f'2022 feature coverage below viewing threshold: {fraction:.3f}')
    profile=dict(driver='GTiff',crs=crs,transform=grid,height=shape[0],width=shape[1],compress='deflate')
    for name,array,order in [('features',features,names),('reflectance',stack,list(BAND_IDS))]:
        with rasterio.open(folder/f'{name}.tif','w',**profile,count=len(array),dtype='float32',nodata=-9999) as dst:
            dst.write(np.where(np.isfinite(array),array,-9999).astype('float32'));dst.descriptions=tuple(order)
    for name,array in [('usable_mask',usable),('study_mask',inside),('scene_classes',scl)]:
        with rasterio.open(folder/f'{name}.tif','w',**profile,count=1,dtype='uint8') as dst:dst.write(array.astype('uint8'),1)
    rgb=(np.moveaxis(np.nan_to_num(np.clip(stack[[2,1,0]]/0.3,0,1)),0,-1)*255).astype('uint8')
    Image.fromarray(np.dstack([rgb,usable.astype('uint8')*255])).save(folder/'preview.png')
    record={'season':'2022','scene_id':item['id'],'acquisition':item['properties']['datetime'],'shape':list(shape),'crs':str(crs),'transform':list(grid)[:6],
            'feature_order':names,'band_order':list(BAND_IDS),'inside_study_pixels':int(inside.sum()),'usable_feature_pixels':int(usable.sum()),
            'usable_fraction_inside_study':fraction,'calibration':calibration,'product_xml_sha256':hashlib.sha256(xml).hexdigest(),
            'calibration_source':item['assets']['product-metadata']['href'],'attribution':'Contains modified Copernicus Sentinel data 2022',
            'source_license_url':'https://cds.climate.copernicus.eu/licences/ec-sentinel','analysis_resolution_m':20,
            'quality_rule':'Valid bands; SCL 4/5/6; valid 3x3 texture neighbourhood; native 20m grid',
            'resampling':'10m reflectance bands averaged to 20m; SCL nearest; 20m bands retained at native resolution'}
    (folder/'report.json').write_bytes((json.dumps(record,indent=2)+'\n').encode())
    (output/'research_report.json').write_bytes((json.dumps({'acquisitions':[record],'cloud_runtime':{'numpy':np.__version__,'rasterio':rasterio.__version__}},indent=2)+'\n').encode())
    print('Verified 2022 crop:',fraction)
    return output


In [ ]:
"""Cloud-only annual crops and weak-reference predictions; no invented imagery."""
import base64
import hashlib
import io
import json
import shutil
import zipfile
from pathlib import Path


def annual_run(study,base_run,feature_builder,model_bytes,expected_sha,output_runner):
    import numpy as np
    import rasterio
    import sklearn
    import joblib
    from sklearn.ensemble import RandomForestClassifier
    if hashlib.sha256(model_bytes).hexdigest()!=expected_sha:raise ValueError('Untrusted model archive')
    with zipfile.ZipFile(io.BytesIO(model_bytes)) as archive:
        manifest=json.loads(archive.read('model_manifest.json'))
        model_available=all(manifest['runtime'][name]==version for name,version in [('numpy',np.__version__),('sklearn',sklearn.__version__),('joblib',joblib.__version__)])
        model=joblib.load(io.BytesIO(archive.read(manifest['selected_model_file']))) if model_available else None
    if model is not None:
        if type(model) is not RandomForestClassifier or model.n_features_in_!=15 or not np.array_equal(model.classes_,[0,1]):raise ValueError('Unsupported annual proxy estimator')
        model.n_jobs=1
    folder=output_runner(study,base_run,feature_builder)
    research=json.loads((folder/'research_report.json').read_bytes());rows=[]
    for record in research['acquisitions']:
        year=int(record['season']);name=str(year)
        with rasterio.open(folder/name/'features.tif') as raster:
            if max(raster.shape)>256 or list(raster.descriptions)!=manifest['feature_order']:raise ValueError('Invalid annual feature grid')
            values=raster.read(masked=True).filled(np.nan);profile=raster.profile.copy()
        with rasterio.open(folder/name/'usable_mask.tif') as raster:valid=raster.read(1)==1
        if not np.isfinite(values[:,valid]).all():raise ValueError('Invalid annual feature values')
        predicted=np.full(valid.shape,255,dtype='uint8')
        if model is not None:
            for start in range(0,valid.shape[0],16):
                mask=valid[start:start+16]
                if mask.any():predicted[start:start+16][mask]=model.predict(values[:,start:start+16][:,mask].T.astype('float32'))
            with rasterio.open(folder/name/'proxy_classes.tif','w',**dict(profile,count=1,dtype='uint8',nodata=255)) as raster:
                raster.write(predicted,1);raster.update_tags(model_kind='exploratory_weak_map_proxy',operational_use_approved='false',acquisition_date=record['acquisition'])
        count=int((predicted==1).sum()) if model is not None else None
        rows.append({'year':year,'date':record['acquisition'][:10],'scene_id':record['scene_id'],
                     'usable_pixels':int(valid.sum()),'study_pixels':record['inside_study_pixels'],
                     'coverage_fraction':record['usable_fraction_inside_study'],
                     'tree_cover_proxy_percent':100*count/int(valid.sum()) if count is not None else None,
                     'tree_cover_proxy_ha':count*.04 if count is not None else None,
                     'median_ndvi':float(np.median(values[record['feature_order'].index('NDVI')][valid])),
                     'model_version':manifest['model_version'] if model is not None else None,
                     'attribution':record['attribution'],'license_url':record['source_license_url'],
                     'canopy_density_percent':None,'synthetic':False})
    report={'format':'forestguard-annual-observations-v1','years_requested':[2022,2023,2024,2025,2026],
            'study_area_version':study['features'][0]['properties']['study_area_version'],
            'observations':sorted(rows,key=lambda row:row['year']),'model_sha256':expected_sha,
            'model_available':model_available,'independent_forest_accuracy_measured':False,
            'limits':'Percentages describe predicted tree-cover class extent within each year\'s clear observed area, not canopy density. Differences in annual percentages are not automatically forest loss; pairwise common coverage and seasonal review are required. Missing images remain missing. Model transfer to these years/seasons has not been independently evaluated.'}
    (folder/'annual_report.json').write_bytes((json.dumps(report,indent=2)+'\n').encode())
    selected=['annual_report.json','research_report.json','boundary.geojson']
    for row in rows:selected.extend(str(row['year'])+'/'+name for name in ['preview.png','features.tif','usable_mask.tif','study_mask.tif','report.json','source.json']+(['proxy_classes.tif'] if model is not None else []))
    for row in rows:
        for name in ['product.xml','scene_classes.tif']:
            if (folder/str(row['year'])/name).exists():selected.append(str(row['year'])+'/'+name)
    hashes={name:hashlib.sha256((folder/name).read_bytes()).hexdigest() for name in selected}
    (folder/'annual_checksums.json').write_bytes((json.dumps(hashes,indent=2)+'\n').encode())
    output=folder/'forestguard_annual_2022_2026.zip'
    with zipfile.ZipFile(output,'w',zipfile.ZIP_DEFLATED) as archive:
        for name in [*selected,'annual_checksums.json']:archive.write(folder/name,name)
    with zipfile.ZipFile(output) as archive:
        if archive.testzip() is not None:raise ValueError('Annual export CRC failure')
        for name,digest in hashes.items():
            if hashlib.sha256(archive.read(name)).hexdigest()!=digest:raise ValueError('Annual export checksum failure')
    # Kaggle's output tree hides deeply nested files by default.
    download=folder.parent.parent/output.name
    shutil.copyfile(output,download)
    print(json.dumps(report,indent=2));print('DOWNLOAD:',download)
    return download


In [ ]:
PC_ITEM={'id': 'S2A_MSIL2A_20221001T052651_R105_T43QFE_20240724T063709', 'bbox': [75.9661585, 21.6063754, 76.8724091, 22.6040892], 'type': 'Feature', 'links': [{'rel': 'collection', 'type': 'application/json', 'href': 'https://planetarycomputer.microsoft.com/api/stac/v1/collections/sentinel-2-l2a'}, {'rel': 'parent', 'type': 'application/json', 'href': 'https://planetarycomputer.microsoft.com/api/stac/v1/collections/sentinel-2-l2a'}, {'rel': 'root', 'type': 'application/json', 'href': 'https://planetarycomputer.microsoft.com/api/stac/v1/'}, {'rel': 'self', 'type': 'application/geo+json', 'href': 'https://planetarycomputer.microsoft.com/api/stac/v1/collections/sentinel-2-l2a/items/S2A_MSIL2A_20221001T052651_R105_T43QFE_20240724T063709'}, {'rel': 'license', 'href': 'https://sentinel.esa.int/documents/247904/690755/Sentinel_Data_Legal_Notice'}, {'rel': 'preview', 'href': 'https://planetarycomputer.microsoft.com/api/data/v1/item/map?collection=sentinel-2-l2a&item=S2A_MSIL2A_20221001T052651_R105_T43QFE_20240724T063709', 'title': 'Map of item', 'type': 'text/html'}], 'assets': {'AOT': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/GRANULE/L2A_T43QFE_A037997_20221001T054202/IMG_DATA/R10m/T43QFE_20221001T052651_AOT_10m.tif', 'proj:bbox': [600000.0, 2390220.0, 709800.0, 2500020.0], 'proj:shape': [10980, 10980], 'proj:transform': [10.0, 0.0, 600000.0, 0.0, -10.0, 2500020.0], 'gsd': 10, 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'roles': ['data'], 'title': 'Aerosol optical thickness (AOT)'}, 'B01': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/GRANULE/L2A_T43QFE_A037997_20221001T054202/IMG_DATA/R60m/T43QFE_20221001T052651_B01_60m.tif', 'proj:bbox': [600000.0, 2390220.0, 709800.0, 2500020.0], 'proj:shape': [1830, 1830], 'proj:transform': [60.0, 0.0, 600000.0, 0.0, -60.0, 2500020.0], 'gsd': 60, 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'roles': ['data'], 'title': 'Band 1 - Coastal aerosol - 60m', 'eo:bands': [{'name': 'B01', 'common_name': 'coastal', 'description': 'Band 1 - Coastal aerosol', 'center_wavelength': 0.443, 'full_width_half_max': 0.027}]}, 'B02': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/GRANULE/L2A_T43QFE_A037997_20221001T054202/IMG_DATA/R10m/T43QFE_20221001T052651_B02_10m.tif', 'proj:bbox': [600000.0, 2390220.0, 709800.0, 2500020.0], 'proj:shape': [10980, 10980], 'proj:transform': [10.0, 0.0, 600000.0, 0.0, -10.0, 2500020.0], 'gsd': 10, 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'roles': ['data'], 'title': 'Band 2 - Blue - 10m', 'eo:bands': [{'name': 'B02', 'common_name': 'blue', 'description': 'Band 2 - Blue', 'center_wavelength': 0.49, 'full_width_half_max': 0.098}]}, 'B03': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/GRANULE/L2A_T43QFE_A037997_20221001T054202/IMG_DATA/R10m/T43QFE_20221001T052651_B03_10m.tif', 'proj:bbox': [600000.0, 2390220.0, 709800.0, 2500020.0], 'proj:shape': [10980, 10980], 'proj:transform': [10.0, 0.0, 600000.0, 0.0, -10.0, 2500020.0], 'gsd': 10, 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'roles': ['data'], 'title': 'Band 3 - Green - 10m', 'eo:bands': [{'name': 'B03', 'common_name': 'green', 'description': 'Band 3 - Green', 'center_wavelength': 0.56, 'full_width_half_max': 0.045}]}, 'B04': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/GRANULE/L2A_T43QFE_A037997_20221001T054202/IMG_DATA/R10m/T43QFE_20221001T052651_B04_10m.tif', 'proj:bbox': [600000.0, 2390220.0, 709800.0, 2500020.0], 'proj:shape': [10980, 10980], 'proj:transform': [10.0, 0.0, 600000.0, 0.0, -10.0, 2500020.0], 'gsd': 10, 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'roles': ['data'], 'title': 'Band 4 - Red - 10m', 'eo:bands': [{'name': 'B04', 'common_name': 'red', 'description': 'Band 4 - Red', 'center_wavelength': 0.665, 'full_width_half_max': 0.038}]}, 'B05': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/GRANULE/L2A_T43QFE_A037997_20221001T054202/IMG_DATA/R20m/T43QFE_20221001T052651_B05_20m.tif', 'proj:bbox': [600000.0, 2390220.0, 709800.0, 2500020.0], 'proj:shape': [5490, 5490], 'proj:transform': [20.0, 0.0, 600000.0, 0.0, -20.0, 2500020.0], 'gsd': 20, 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'roles': ['data'], 'title': 'Band 5 - Vegetation red edge 1 - 20m', 'eo:bands': [{'name': 'B05', 'common_name': 'rededge', 'description': 'Band 5 - Vegetation red edge 1', 'center_wavelength': 0.704, 'full_width_half_max': 0.019}]}, 'B06': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/GRANULE/L2A_T43QFE_A037997_20221001T054202/IMG_DATA/R20m/T43QFE_20221001T052651_B06_20m.tif', 'proj:bbox': [600000.0, 2390220.0, 709800.0, 2500020.0], 'proj:shape': [5490, 5490], 'proj:transform': [20.0, 0.0, 600000.0, 0.0, -20.0, 2500020.0], 'gsd': 20, 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'roles': ['data'], 'title': 'Band 6 - Vegetation red edge 2 - 20m', 'eo:bands': [{'name': 'B06', 'common_name': 'rededge', 'description': 'Band 6 - Vegetation red edge 2', 'center_wavelength': 0.74, 'full_width_half_max': 0.018}]}, 'B07': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/GRANULE/L2A_T43QFE_A037997_20221001T054202/IMG_DATA/R20m/T43QFE_20221001T052651_B07_20m.tif', 'proj:bbox': [600000.0, 2390220.0, 709800.0, 2500020.0], 'proj:shape': [5490, 5490], 'proj:transform': [20.0, 0.0, 600000.0, 0.0, -20.0, 2500020.0], 'gsd': 20, 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'roles': ['data'], 'title': 'Band 7 - Vegetation red edge 3 - 20m', 'eo:bands': [{'name': 'B07', 'common_name': 'rededge', 'description': 'Band 7 - Vegetation red edge 3', 'center_wavelength': 0.783, 'full_width_half_max': 0.028}]}, 'B08': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/GRANULE/L2A_T43QFE_A037997_20221001T054202/IMG_DATA/R10m/T43QFE_20221001T052651_B08_10m.tif', 'proj:bbox': [600000.0, 2390220.0, 709800.0, 2500020.0], 'proj:shape': [10980, 10980], 'proj:transform': [10.0, 0.0, 600000.0, 0.0, -10.0, 2500020.0], 'gsd': 10, 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'roles': ['data'], 'title': 'Band 8 - NIR - 10m', 'eo:bands': [{'name': 'B08', 'common_name': 'nir', 'description': 'Band 8 - NIR', 'center_wavelength': 0.842, 'full_width_half_max': 0.145}]}, 'B09': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/GRANULE/L2A_T43QFE_A037997_20221001T054202/IMG_DATA/R60m/T43QFE_20221001T052651_B09_60m.tif', 'proj:bbox': [600000.0, 2390220.0, 709800.0, 2500020.0], 'proj:shape': [1830, 1830], 'proj:transform': [60.0, 0.0, 600000.0, 0.0, -60.0, 2500020.0], 'gsd': 60, 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'roles': ['data'], 'title': 'Band 9 - Water vapor - 60m', 'eo:bands': [{'name': 'B09', 'description': 'Band 9 - Water vapor', 'center_wavelength': 0.945, 'full_width_half_max': 0.026}]}, 'B11': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/GRANULE/L2A_T43QFE_A037997_20221001T054202/IMG_DATA/R20m/T43QFE_20221001T052651_B11_20m.tif', 'proj:bbox': [600000.0, 2390220.0, 709800.0, 2500020.0], 'proj:shape': [5490, 5490], 'proj:transform': [20.0, 0.0, 600000.0, 0.0, -20.0, 2500020.0], 'gsd': 20, 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'roles': ['data'], 'title': 'Band 11 - SWIR (1.6) - 20m', 'eo:bands': [{'name': 'B11', 'common_name': 'swir16', 'description': 'Band 11 - SWIR (1.6)', 'center_wavelength': 1.61, 'full_width_half_max': 0.143}]}, 'B12': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/GRANULE/L2A_T43QFE_A037997_20221001T054202/IMG_DATA/R20m/T43QFE_20221001T052651_B12_20m.tif', 'proj:bbox': [600000.0, 2390220.0, 709800.0, 2500020.0], 'proj:shape': [5490, 5490], 'proj:transform': [20.0, 0.0, 600000.0, 0.0, -20.0, 2500020.0], 'gsd': 20, 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'roles': ['data'], 'title': 'Band 12 - SWIR (2.2) - 20m', 'eo:bands': [{'name': 'B12', 'common_name': 'swir22', 'description': 'Band 12 - SWIR (2.2)', 'center_wavelength': 2.19, 'full_width_half_max': 0.242}]}, 'B8A': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/GRANULE/L2A_T43QFE_A037997_20221001T054202/IMG_DATA/R20m/T43QFE_20221001T052651_B8A_20m.tif', 'proj:bbox': [600000.0, 2390220.0, 709800.0, 2500020.0], 'proj:shape': [5490, 5490], 'proj:transform': [20.0, 0.0, 600000.0, 0.0, -20.0, 2500020.0], 'gsd': 20, 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'roles': ['data'], 'title': 'Band 8A - Vegetation red edge 4 - 20m', 'eo:bands': [{'name': 'B8A', 'common_name': 'rededge', 'description': 'Band 8A - Vegetation red edge 4', 'center_wavelength': 0.865, 'full_width_half_max': 0.033}]}, 'SCL': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/GRANULE/L2A_T43QFE_A037997_20221001T054202/IMG_DATA/R20m/T43QFE_20221001T052651_SCL_20m.tif', 'proj:bbox': [600000.0, 2390220.0, 709800.0, 2500020.0], 'proj:shape': [5490, 5490], 'proj:transform': [20.0, 0.0, 600000.0, 0.0, -20.0, 2500020.0], 'gsd': 20, 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'roles': ['data'], 'title': 'Scene classfication map (SCL)'}, 'WVP': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/GRANULE/L2A_T43QFE_A037997_20221001T054202/IMG_DATA/R10m/T43QFE_20221001T052651_WVP_10m.tif', 'proj:bbox': [600000.0, 2390220.0, 709800.0, 2500020.0], 'proj:shape': [10980, 10980], 'proj:transform': [10.0, 0.0, 600000.0, 0.0, -10.0, 2500020.0], 'gsd': 10, 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'roles': ['data'], 'title': 'Water vapour (WVP)'}, 'visual': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/GRANULE/L2A_T43QFE_A037997_20221001T054202/IMG_DATA/R10m/T43QFE_20221001T052651_TCI_10m.tif', 'proj:bbox': [600000.0, 2390220.0, 709800.0, 2500020.0], 'proj:shape': [10980, 10980], 'proj:transform': [10.0, 0.0, 600000.0, 0.0, -10.0, 2500020.0], 'gsd': 10, 'type': 'image/tiff; application=geotiff; profile=cloud-optimized', 'roles': ['data'], 'title': 'True color image', 'eo:bands': [{'name': 'B04', 'common_name': 'red', 'description': 'Band 4 - Red', 'center_wavelength': 0.665, 'full_width_half_max': 0.038}, {'name': 'B03', 'common_name': 'green', 'description': 'Band 3 - Green', 'center_wavelength': 0.56, 'full_width_half_max': 0.045}, {'name': 'B02', 'common_name': 'blue', 'description': 'Band 2 - Blue', 'center_wavelength': 0.49, 'full_width_half_max': 0.098}]}, 'safe-manifest': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/manifest.safe', 'type': 'application/xml', 'roles': ['metadata'], 'title': 'SAFE manifest'}, 'granule-metadata': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/GRANULE/L2A_T43QFE_A037997_20221001T054202/MTD_TL.xml', 'type': 'application/xml', 'roles': ['metadata'], 'title': 'Granule metadata'}, 'inspire-metadata': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/INSPIRE.xml', 'type': 'application/xml', 'roles': ['metadata'], 'title': 'INSPIRE metadata'}, 'product-metadata': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/MTD_MSIL2A.xml', 'type': 'application/xml', 'roles': ['metadata'], 'title': 'Product metadata'}, 'datastrip-metadata': {'href': 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/Q/FE/2022/10/01/S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE/DATASTRIP/DS_S2RP_20240724T063709_S20221001T054202/MTD_DS.xml', 'type': 'application/xml', 'roles': ['metadata'], 'title': 'Datastrip metadata'}, 'tilejson': {'title': 'TileJSON with default rendering', 'href': 'https://planetarycomputer.microsoft.com/api/data/v1/item/tilejson.json?collection=sentinel-2-l2a&item=S2A_MSIL2A_20221001T052651_R105_T43QFE_20240724T063709&assets=visual&asset_bidx=visual%7C1,2,3&nodata=0&format=png', 'type': 'application/json', 'roles': ['tiles']}, 'rendered_preview': {'title': 'Rendered preview', 'rel': 'preview', 'href': 'https://planetarycomputer.microsoft.com/api/data/v1/item/preview.png?collection=sentinel-2-l2a&item=S2A_MSIL2A_20221001T052651_R105_T43QFE_20240724T063709&assets=visual&asset_bidx=visual%7C1,2,3&nodata=0&format=png', 'roles': ['overview'], 'type': 'image/png'}}, 'geometry': {'type': 'Polygon', 'coordinates': [[[76.8724091, 22.5956461], [76.8620353, 22.555165], [76.824143, 22.4073032], [76.7862444, 22.2593885], [76.7483673, 22.1113896], [76.7104267, 21.9633667], [76.6725381, 21.8152694], [76.6346082, 21.6670768], [76.6191371, 21.6063754], [75.9661585, 21.6122496], [75.972938, 22.6040892], [76.8724091, 22.5956461]]]}, 'collection': 'sentinel-2-l2a', 'properties': {'datetime': '2022-10-01T05:26:51.024000Z', 'platform': 'Sentinel-2A', 'proj:epsg': 32643, 'instruments': ['msi'], 's2:mgrs_tile': '43QFE', 'constellation': 'Sentinel 2', 's2:granule_id': 'S2A_OPER_MSI_L2A_TL_S2RP_20240724T063709_A037997_T43QFE_N05.10', 'eo:cloud_cover': 2.092795, 's2:datatake_id': 'GS2A_20221001T052651_037997_N05.10', 's2:product_uri': 'S2A_MSIL2A_20221001T052651_N0510_R105_T43QFE_20240724T063709.SAFE', 's2:datastrip_id': 'S2A_OPER_MSI_L2A_DS_S2RP_20240724T063709_S20221001T054202_N05.10', 's2:product_type': 'S2MSI2A', 'sat:orbit_state': 'descending', 's2:datatake_type': 'INS-NOBS', 's2:generation_time': '2024-07-24T06:37:09.000000Z', 'sat:relative_orbit': 105, 's2:water_percentage': 7.289725, 's2:mean_solar_zenith': 29.3371187718007, 's2:mean_solar_azimuth': 147.704462518936, 's2:processing_baseline': '05.10', 's2:snow_ice_percentage': 0.0, 's2:vegetation_percentage': 75.750041, 's2:thin_cirrus_percentage': 1.811885, 's2:cloud_shadow_percentage': 0.333263, 's2:nodata_pixel_percentage': 27.08202, 's2:unclassified_percentage': 2.38861, 's2:dark_features_percentage': 0.35392, 's2:not_vegetated_percentage': 11.791646, 's2:degraded_msi_data_percentage': 0.0295, 's2:high_proba_clouds_percentage': 0.079613, 's2:reflectance_conversion_factor': 0.996018746323769, 's2:medium_proba_clouds_percentage': 0.201296, 's2:saturated_defective_pixel_percentage': 0.0}, 'stac_extensions': ['https://stac-extensions.github.io/eo/v1.0.0/schema.json', 'https://stac-extensions.github.io/sat/v1.0.0/schema.json', 'https://stac-extensions.github.io/projection/v1.0.0/schema.json'], 'stac_version': '1.0.0'}
annual_output=annual_run(STUDY,None,research_features,MODEL_BYTES,MODEL_SHA,run_2022)
